# RSNA knee: frozen Pillar-0 experiment, data gate
Private first run: inventory competition data, sample DICOM headers, reconstruct up to
three volumes. All competition files and derived artifacts remain on Kaggle. No full
preprocessing, encoder substitution, pseudo-label generation or training is performed.
The next experiment step depends on the actual audit and visual geometry review.


In [ ]:
import sys, subprocess, json, os
from pathlib import Path
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pydicom==3.0.1'], check=True)
PROJECT = Path('/kaggle/working/project')
SOURCE_FILES = {'src/rsna_knee/__init__.py': '"""Frozen volumetric MRI transfer experiment."""\n', 'src/rsna_knee/data/__init__.py': '"""Dataset inventory without inferred supervision."""\n', 'src/rsna_knee/data/audit.py': '"""Inventory all filenames/CSV rows, sample DICOM series headers; never decode pixels."""\nimport argparse\nfrom collections import Counter, defaultdict\nimport csv\nimport hashlib\nimport json\nfrom pathlib import Path\nimport random\nimport sys\nimport platform\n\nimport numpy as np\nimport pydicom\n\nfrom rsna_knee.preprocessing.dicom import geometry\n\nFIELDS = (\'PatientID\', \'StudyInstanceUID\', \'SeriesInstanceUID\', \'SOPInstanceUID\',\n          \'Modality\', \'SeriesDescription\', \'ProtocolName\', \'ScanningSequence\',\n          \'SequenceVariant\', \'ScanOptions\', \'ImageType\', \'EchoTime\', \'RepetitionTime\',\n          \'InversionTime\', \'MagneticFieldStrength\', \'Manufacturer\', \'ManufacturerModelName\',\n          \'InstitutionName\', \'Rows\', \'Columns\', \'PixelSpacing\', \'SliceThickness\',\n          \'SpacingBetweenSlices\', \'ImageOrientationPatient\', \'ImagePositionPatient\',\n          \'InstanceNumber\', \'AcquisitionNumber\', \'TemporalPositionIdentifier\', \'NumberOfFrames\')\n\n\ndef digest_file(path):\n    h = hashlib.sha256()\n    with path.open(\'rb\') as stream:\n        for chunk in iter(lambda: stream.read(1024 * 1024), b\'\'):\n            h.update(chunk)\n    return h.hexdigest()\n\n\ndef csv_audit(path, study_column=None, targets=()):\n    missing, values, ids, hashes = Counter(), defaultdict(Counter), Counter(), Counter()\n    n = 0\n    with path.open(newline=\'\', encoding=\'utf-8-sig\') as stream:\n        reader = csv.DictReader(stream)\n        columns = reader.fieldnames or []\n        if study_column and study_column not in columns:\n            raise ValueError(f\'{path}: missing study column {study_column}\')\n        if set(targets) - set(columns):\n            raise ValueError(f\'{path}: missing target columns\')\n        for row in reader:\n            n += 1\n            hashes[hashlib.sha256(json.dumps(row, sort_keys=True).encode()).hexdigest()] += 1\n            for key in columns:\n                value = row[key]\n                if value is None or value.strip().lower() in (\'\', \'nan\', \'null\', \'none\', \'na\'):\n                    missing[key] += 1\n                elif key in targets:\n                    values[key][value] += 1\n            if study_column and row[study_column]:\n                ids[row[study_column]] += 1\n    return {\'rows\': n, \'columns\': columns, \'missing\': dict(missing),\n            \'duplicate_rows\': sum(v - 1 for v in hashes.values()),\n            \'unique_studies\': len(ids) if study_column else None,\n            \'duplicate_study_rows\': sum(v - 1 for v in ids.values()) if study_column else None,\n            \'target_values\': dict(values), \'sha256\': digest_file(path)}\n\n\ndef audit(root, config):\n    if not root.is_dir():\n        raise ValueError(f\'Dataset directory does not exist: {root}\')\n    groups, extensions, tables, archives = defaultdict(list), Counter(), {}, []\n    for path in sorted(root.rglob(\'*\')):\n        if not path.is_file():\n            continue\n        rel = path.relative_to(root)\n        extensions[path.suffix.lower()] += 1\n        if path.suffix.lower() in config[\'dicom_suffixes\']:\n            groups[str(rel.parent)].append(path)\n        elif path.suffix.lower() == \'.csv\':\n            is_labels = str(rel) == config.get(\'labels_csv\')\n            tables[str(rel)] = csv_audit(path, config.get(\'study_column\') if is_labels else None,\n                                        config.get(\'target_columns\', []) if is_labels else [])\n        elif path.suffix.lower() in (\'.zip\', \'.gz\', \'.tar\', \'.7z\'):\n            archives.append(str(rel))\n    rng = random.Random(config[\'seed\'])\n    chosen = sorted(rng.sample(sorted(groups), min(config[\'sample_series\'], len(groups))))\n    sampled, seen_sop = [], Counter()\n    for directory in chosen:\n        entry = {\'directory\': directory, \'files\': len(groups[directory])}\n        try:\n            headers = [pydicom.dcmread(p, stop_before_pixels=True) for p in groups[directory]]\n            entry[\'fields_available\'] = sorted({d.keyword for h in headers for d in h if d.keyword})\n            entry[\'missing_fields\'] = {key: sum(not hasattr(h, key) for h in headers) for key in FIELDS}\n            entry[\'values\'] = {key: dict(Counter(str(getattr(h, key, \'\')) for h in headers)) for key in FIELDS}\n            seen_sop.update(str(h.SOPInstanceUID) for h in headers if hasattr(h, \'SOPInstanceUID\'))\n            _, meta = geometry(headers)\n            entry[\'geometry\'] = meta\n            entry[\'status\'] = \'valid_geometry\'\n        except Exception as exc:\n            entry[\'status\'] = \'flagged\'\n            entry[\'error\'] = f\'{type(exc).__name__}: {exc}\'\n        sampled.append(entry)\n    # Directory names are candidates only until checked against DICOM UIDs.\n    studies = defaultdict(list)\n    for directory, paths in groups.items():\n        studies[str(Path(directory).parent)].append(len(paths))\n    inventory = [{\'study_directory_candidate\': key, \'series_directories\': len(v),\n                  \'slice_file_counts\': v} for key, v in sorted(studies.items())]\n    return {\n        \'status\': \'sampled_audit\' if groups else \'no_dicom_found\',\n        \'root\': str(root), \'config\': config,\n        \'software\': {\'python\': platform.python_version(), \'numpy\': np.__version__, \'pydicom\': pydicom.__version__},\n        \'file_extensions\': dict(extensions), \'archives_not_opened\': archives, \'tables\': tables,\n        \'directory_inventory\': inventory, \'dicom_series_directory_candidates\': len(groups),\n        \'sampled_series\': sampled, \'sample_duplicate_sop_uids\': sum(v - 1 for v in seen_sop.values()),\n        \'unverified\': [\'Directory hierarchy is not authoritative study/series identity.\',\n                       \'Train/test counts and labelled/unlabelled joins require reviewed table schema.\',\n                       \'Patient IDs may be missing or anonymized; no safe split approved yet.\',\n                       \'Site/protocol distribution is sampled; no leakage conclusion yet.\',\n                       \'Fluid sensitivity/fat suppression require sequence review; no text heuristics applied.\',\n                       \'Reports remain text; target names and metric require competition verification.\',\n                       \'Only configured suffixes inventoried as DICOM; inspect extension counts for others.\'],\n    }\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'root\', type=Path)\n    parser.add_argument(\'--config\', type=Path, default=Path(\'configs/audit.json\'))\n    parser.add_argument(\'--output\', type=Path, required=True)\n    args = parser.parse_args()\n    config = json.loads(args.config.read_text())\n    if config[\'sample_series\'] < 1:\n        parser.error(\'sample_series must be positive\')\n    result = audit(args.root, config)\n    args.output.mkdir(parents=True, exist_ok=True)\n    (args.output / \'audit.json\').write_text(json.dumps(result, indent=2))\n    lines = [\'# Dataset audit\', \'\', f\'Status: {result["status"]}\',\n             f\'DICOM series directory candidates: {result["dicom_series_directory_candidates"]}\',\n             f\'Series sampled: {len(result["sampled_series"])}\', \'\', \'## Tables\', \'\']\n    lines += [f\'- {key}: {value["rows"]} rows; columns: {value["columns"]}\' for key, value in result[\'tables\'].items()]\n    lines += [\'\', \'## Not yet verified\', \'\'] + [f\'- {x}\' for x in result[\'unverified\']]\n    (args.output / \'AUDIT.md\').write_text(\'\\n\'.join(lines) + \'\\n\')\n    print(f\'Audit written to {args.output}; status={result["status"]}\')\n    if not result[\'dicom_series_directory_candidates\']:\n        sys.exit(2)\n\n\nif __name__ == \'__main__\':\n    main()\n', 'src/rsna_knee/preprocessing/__init__.py': '"""Geometry-preserving DICOM reconstruction."""\n', 'src/rsna_knee/preprocessing/dicom.py': '"""Classic single-frame MR reconstruction. Array axes are (slice, row, column)."""\nimport argparse\nimport json\nfrom pathlib import Path\n\nimport numpy as np\nimport pydicom\n\n\nclass GeometryError(ValueError):\n    """Unsupported or inconsistent geometry; never fall back to filenames."""\n\n\ndef geometry(datasets):\n    if len(datasets) < 2:\n        raise GeometryError(\'At least two slices required to establish physical spacing\')\n    required = (\'ImagePositionPatient\', \'ImageOrientationPatient\', \'PixelSpacing\',\n                \'Rows\', \'Columns\', \'StudyInstanceUID\', \'SeriesInstanceUID\', \'SOPInstanceUID\')\n    for d in datasets:\n        if any(not hasattr(d, key) for key in required):\n            raise GeometryError(\'Missing required geometry or UID fields\')\n        if getattr(d, \'Modality\', \'\') != \'MR\' or int(getattr(d, \'NumberOfFrames\', 1)) != 1:\n            raise GeometryError(\'Only single-frame MR is supported; separate enhanced MR explicitly\')\n    for key in (\'StudyInstanceUID\', \'SeriesInstanceUID\', \'Rows\', \'Columns\'):\n        if len({str(getattr(d, key)) for d in datasets}) != 1:\n            raise GeometryError(f\'Inconsistent {key}\')\n    if len({str(d.SOPInstanceUID) for d in datasets}) != len(datasets):\n        raise GeometryError(\'Duplicate SOPInstanceUID\')\n    iop = np.asarray(datasets[0].ImageOrientationPatient, dtype=float)\n    spacing = np.asarray(datasets[0].PixelSpacing, dtype=float)\n    positions = np.asarray([d.ImagePositionPatient for d in datasets], dtype=float)\n    if iop.shape != (6,) or spacing.shape != (2,) or positions.shape != (len(datasets), 3):\n        raise GeometryError(\'Invalid geometry dimensions\')\n    if not all(np.isfinite(x).all() for x in (iop, spacing, positions)) or (spacing <= 0).any():\n        raise GeometryError(\'Nonfinite geometry or nonpositive pixel spacing\')\n    col_direction, row_direction = iop[:3], iop[3:]\n    if not np.allclose([np.linalg.norm(col_direction), np.linalg.norm(row_direction),\n                        np.dot(col_direction, row_direction)], [1, 1, 0], atol=1e-4):\n        raise GeometryError(\'Orientation cosines are not orthonormal\')\n    for d in datasets:\n        if not np.allclose(d.ImageOrientationPatient, iop, atol=1e-4, rtol=0):\n            raise GeometryError(\'Varying orientation (localizer or mixed acquisition)\')\n        if not np.allclose(d.PixelSpacing, spacing, atol=1e-4, rtol=0):\n            raise GeometryError(\'Varying pixel spacing\')\n    for key in (\'EchoTime\', \'TemporalPositionIdentifier\', \'AcquisitionNumber\'):\n        if len({str(getattr(d, key, \'\')) for d in datasets}) > 1:\n            raise GeometryError(f\'Mixed acquisition: {key}\')\n    normal = np.cross(col_direction, row_direction)\n    order = np.argsort(positions @ normal, kind=\'stable\')\n    positions = positions[order]\n    gaps = np.diff(positions @ normal)\n    dz = float(np.median(gaps))\n    if (gaps < 1e-3).any():\n        raise GeometryError(\'Repeated slice positions; split echoes/timepoints explicitly\')\n    if not np.allclose(gaps, dz, atol=0.05, rtol=0.01):\n        raise GeometryError(\'Irregular slice spacing or missing slices\')\n    if not np.allclose(np.diff(positions, axis=0), normal * dz, atol=0.05, rtol=0.01):\n        raise GeometryError(\'In-plane slice displacement; unsupported sheared volume\')\n    affine = np.eye(4)\n    affine[:3, 0] = normal * dz\n    affine[:3, 1] = row_direction * spacing[0]\n    affine[:3, 2] = col_direction * spacing[1]\n    affine[:3, 3] = positions[0]\n    plane = [\'sagittal\', \'coronal\', \'axial\'][int(np.argmax(np.abs(normal)))]\n    if np.max(np.abs(normal)) < 0.95:\n        plane = \'oblique\'\n    metadata = {\n        \'array_axes\': [\'slice\', \'row\', \'column\'], \'world_coordinates\': \'DICOM LPS (mm)\',\n        \'affine_array_to_lps\': affine.tolist(), \'spacing_mm\': [dz, *spacing.tolist()],\n        \'normal_lps\': normal.tolist(), \'plane\': plane, \'slice_count\': len(datasets),\n        \'positions_lps\': positions.tolist(), \'orientation_patient\': iop.tolist(),\n        \'study_uid\': str(datasets[0].StudyInstanceUID),\n        \'series_uid\': str(datasets[0].SeriesInstanceUID),\n        \'series_description\': str(getattr(datasets[0], \'SeriesDescription\', \'\')),\n        \'slice_thickness\': str(getattr(datasets[0], \'SliceThickness\', \'\')),\n        \'spacing_between_slices\': str(getattr(datasets[0], \'SpacingBetweenSlices\', \'\')),\n        \'transformations\': [\'sort by IPP dot cross(IOP[:3], IOP[3:])\',\n                            \'apply per-slice RescaleSlope and RescaleIntercept\'],\n    }\n    return order, metadata\n\n\ndef reconstruct(paths):\n    paths = list(map(Path, paths))\n    headers = [pydicom.dcmread(p, stop_before_pixels=True) for p in paths]\n    order, metadata = geometry(headers)\n    slices = []\n    for index in order:\n        d = pydicom.dcmread(paths[index])\n        if int(getattr(d, \'SamplesPerPixel\', 1)) != 1 or getattr(d, \'PhotometricInterpretation\', \'\') != \'MONOCHROME2\':\n            raise GeometryError(\'Only MONOCHROME2 grayscale MR is supported\')\n        if hasattr(d, \'ModalityLUTSequence\') or hasattr(d, \'RealWorldValueMappingSequence\'):\n            raise GeometryError(\'Explicit LUT/real-world mapping requires a reviewed conversion\')\n        pixels = d.pixel_array.astype(np.float32)\n        if pixels.shape != (int(d.Rows), int(d.Columns)):\n            raise GeometryError(\'Pixel shape differs from header\')\n        if hasattr(d, \'PixelPaddingValue\'):\n            low = min(float(d.PixelPaddingValue), float(getattr(d, \'PixelPaddingRangeLimit\', d.PixelPaddingValue)))\n            high = max(float(d.PixelPaddingValue), float(getattr(d, \'PixelPaddingRangeLimit\', d.PixelPaddingValue)))\n            pixels[(pixels >= low) & (pixels <= high)] = np.nan\n        pixels = pixels * float(getattr(d, \'RescaleSlope\', 1)) + float(getattr(d, \'RescaleIntercept\', 0))\n        if not np.isfinite(pixels).any():\n            raise GeometryError(\'No finite non-padding pixels in slice\')\n        slices.append(pixels)\n    metadata[\'ordered_files\'] = [str(paths[i]) for i in order]\n    metadata[\'padding_policy\'] = \'DICOM padding becomes NaN; excluded from display percentiles\'\n    return np.stack(slices), metadata\n\n\ndef visualize(volume, metadata, output):\n    import matplotlib\n    matplotlib.use(\'Agg\')\n    import matplotlib.pyplot as plt\n    finite = volume[np.isfinite(volume)]\n    lo, hi = np.percentile(finite, [1, 99])\n    dz, dy, dx = metadata[\'spacing_mm\']\n    fig, axes = plt.subplots(2, 3, figsize=(12, 8))\n    for ax, index in zip(axes[0], np.linspace(0, len(volume) - 1, 3).astype(int)):\n        ax.imshow(volume[index], cmap=\'gray\', vmin=lo, vmax=hi, aspect=dy / dx)\n        ax.set_title(f\'Slice {index}; LPS {np.round(metadata["positions_lps"][index], 1)}\')\n        ax.set_xlabel(\'column\'); ax.set_ylabel(\'row\')\n    views = [(volume[:, volume.shape[1] // 2, :], dz / dx, \'column\'),\n             (volume[:, :, volume.shape[2] // 2], dz / dy, \'row\')]\n    for ax, (pixels, aspect, xlabel) in zip(axes[1], views):\n        ax.imshow(pixels, cmap=\'gray\', vmin=lo, vmax=hi, aspect=aspect)\n        ax.set_xlabel(xlabel); ax.set_ylabel(\'physically ordered slice\')\n    axes[1, 2].axis(\'off\')\n    axes[1, 2].text(0, 0.5, f\'{metadata["plane"]}\\nSpacing z,y,x: {dz:g}, {dy:g}, {dx:g} mm\\n\'\n                           \'Native acquisition axes; not canonical anatomy.\\nDisplay only: 1–99% intensity window.\\nNo resampling, cropping or normalization.\', wrap=True)\n    fig.tight_layout(); fig.savefig(output, dpi=140); plt.close(fig)\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'series\', type=Path)\n    parser.add_argument(\'--output\', type=Path, required=True)\n    args = parser.parse_args()\n    paths = sorted(p for p in args.series.iterdir() if p.is_file())\n    volume, metadata = reconstruct(paths)\n    args.output.mkdir(parents=True, exist_ok=True)\n    np.save(args.output / \'volume.npy\', volume)\n    (args.output / \'geometry.json\').write_text(json.dumps(metadata, indent=2))\n    visualize(volume, metadata, args.output / \'views.png\')\n\n\nif __name__ == \'__main__\':\n    main()\n', 'configs/audit.json': '{\n  "seed": 42,\n  "sample_series": 32,\n  "dicom_suffixes": [".dcm", ".dicom"],\n  "labels_csv": null,\n  "study_column": null,\n  "target_columns": [],\n  "dataset_version": "unverified"\n}\n'}
for name, source in SOURCE_FILES.items():
    dest = PROJECT / name
    dest.parent.mkdir(parents=True, exist_ok=True)
    dest.write_text(source)
sys.path.insert(0, str(PROJECT / 'src'))
print('Project source installed; no competition files copied.')


In [ ]:
from rsna_knee.data.audit import audit
mount = Path('/kaggle/input')
roots = [p for p in [mount / 'rsna-knee-abnormality-detection', mount / 'competitions/rsna-knee-abnormality-detection'] if p.is_dir()]
if len(roots) != 1:
    raise RuntimeError(f'Expected one competition mount, found {len(roots)}; inspect mounted inputs')
DATA = roots[0]
OUTPUT = Path('/kaggle/working/knee-audit')
OUTPUT.mkdir(exist_ok=True)
config = json.loads((PROJECT / 'configs/audit.json').read_text())
config['sample_series'] = 32
result = audit(DATA, config)
(OUTPUT / 'audit.json').write_text(json.dumps(result, indent=2))
# Operational log contains aggregates/schema only, no individual identifiers or reports.
summary = {k: result[k] for k in ['status', 'file_extensions', 'dicom_series_directory_candidates', 'sample_duplicate_sop_uids']}
summary['tables'] = result['tables']
summary['geometry_status'] = dict(__import__('collections').Counter(x['status'] for x in result['sampled_series']))
summary['geometry_errors'] = dict(__import__('collections').Counter(x.get('error', '') for x in result['sampled_series'] if x['status'] != 'valid_geometry'))
summary['planes'] = dict(__import__('collections').Counter(x['geometry']['plane'] for x in result['sampled_series'] if x['status'] == 'valid_geometry'))
print('AUDIT_SUMMARY_BEGIN')
print(json.dumps(summary, indent=2))
print('AUDIT_SUMMARY_END')
if result['status'] != 'sampled_audit':
    raise RuntimeError('No DICOM recognized: inspect extension inventory before proceeding')


In [ ]:
import numpy as np
from rsna_knee.preprocessing.dicom import reconstruct, visualize
from IPython.display import Image, display
valid = [x for x in result['sampled_series'] if x['status'] == 'valid_geometry']
completed = 0
for index, item in enumerate(valid[:3]):
    paths = sorted(p for p in (DATA / item['directory']).iterdir() if p.suffix.lower() in config['dicom_suffixes'])
    try:
        volume, metadata = reconstruct(paths)
        dest = OUTPUT / f'volume-{index}'
        dest.mkdir(exist_ok=True)
        np.save(dest / 'volume.npy', volume)
        (dest / 'geometry.json').write_text(json.dumps(metadata, indent=2))
        visualize(volume, metadata, dest / 'views.png')
        display(Image(filename=str(dest / 'views.png')))
        print('VOLUME_OK', index, list(volume.shape), metadata['plane'], metadata['spacing_mm'])
        completed += 1
    except Exception as exc:
        (OUTPUT / f'volume-{index}-error.json').write_text(json.dumps({'error': f'{type(exc).__name__}: {exc}'}))
        print('VOLUME_FAILED', index, type(exc).__name__)
print('RECONSTRUCTED_VOLUMES', completed)
if completed == 0:
    raise RuntimeError('No reconstructed volumes; resolve audit/decoder failures first')


## Next gate
Review private audit artifacts and real-volume images here on Kaggle. Verify official
label schema, patient grouping and metric. Then pin and load the gated
`YalaLab/Pillar0-BreastMRI` checkpoint with frozen weights and reviewed preprocessing.
No end-to-end classification result is claimed by this audit notebook.
